[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/nilay47/marginally-realistic/blob/main/notebooks/seeds_colab.ipynb)

# Marginally Realistic, Jointly Impossible: seeds 1 and 2
Runs two extra 30k-step seeds under the seed-0 protocol, then builds the report and figure. Results go to Google Drive, so a rerun after a disconnect resumes. Run top to bottom.

## 1. GPU check
Stops if no GPU. Use Runtime → Change runtime type → GPU.

In [ ]:
import subprocess, torch
print(subprocess.run(["nvidia-smi", "-L"], capture_output=True, text=True).stdout or "nvidia-smi: no output")
if not torch.cuda.is_available():
    raise SystemExit("No GPU: switch to a GPU runtime (Runtime -> Change runtime type -> GPU) and rerun.")
print("cuda available:", torch.cuda.get_device_name(0))

## 2. Mount Google Drive
Outputs are written to `OUT` on Drive.

In [ ]:
from google.colab import drive
drive.mount("/content/drive")
OUT = "/content/drive/MyDrive/physgen_seeds"
import os; os.makedirs(OUT, exist_ok=True); os.environ["OUT"] = OUT

## 3. Get the code
Pulls if `/content/repo` exists, otherwise clones.

In [ ]:
import os
if os.path.exists("/content/repo"):
    !git -C /content/repo pull
else:
    !git clone https://github.com/nilay47/marginally-realistic.git /content/repo
%cd /content/repo

## 4. Install
Only `scipy` and `matplotlib`; torch is already on Colab.

In [ ]:
!pip install -q scipy matplotlib

## 5. Sanity check
Runs `sim.py` and fails loudly if the real-data `r_energy` floor is not below 1e-4 (that would mean the pendulum `sim.py` is loaded).

In [ ]:
import re, subprocess
out = subprocess.run(["python", "sim.py"], capture_output=True, text=True).stdout
print(out)
m = re.search(r"median \|r_energy\|\s*=\s*([0-9.eE+-]+)", out)
assert m and float(m.group(1)) < 1e-4, f"r_energy floor {m and m.group(1)} is not < 1e-4: wrong sim.py loaded"
print("OK: r_energy floor", m.group(1))

## 6. Seed 1
Skipped if `seed1/log.json` already exists on Drive. Progress streams live and is saved to `seed1.log`.

In [ ]:
%%bash
set -o pipefail
if [ -f "$OUT/seed1/log.json" ]; then echo "seed 1 already done, skipping"; else
  python -u train.py --seed 1 --out "$OUT" 2>&1 | tee "$OUT/seed1.log"
fi

## 7. Seed 2
Same as seed 1.

In [ ]:
%%bash
set -o pipefail
if [ -f "$OUT/seed2/log.json" ]; then echo "seed 2 already done, skipping"; else
  python -u train.py --seed 2 --out "$OUT" 2>&1 | tee "$OUT/seed2.log"
fi

## 8. Report
Writes `seeds_report.json` and the figure to `OUT`, prints the table, shows the figure. The bootstrap takes a few minutes.

In [ ]:
import subprocess
from IPython.display import Markdown, Image, display
r = subprocess.run(["python", "seeds_report.py", "--out", OUT], capture_output=True, text=True)
print(r.stderr[-2000:])
display(Markdown(r.stdout))
display(Image(f"{OUT}/fig_training_3seeds.png"))

## 9. Download (optional)
Zips the report, figure and logs.

In [ ]:
from google.colab import files
!cd "$OUT" && zip -q seeds_out.zip seeds_report.json fig_training_3seeds.pdf fig_training_3seeds.png seed1.log seed2.log seed1/log.json seed2/log.json
files.download(f"{OUT}/seeds_out.zip")